# Intelli-Speech — AUDIT_V2 (Experiment 0: read-only dataset audit)

Verifies the facts every v2 experiment depends on, before any analysis or synthesis:

- Vaani Atypical Speech Corpus, **Telugu config only** (gated; research use only).
- Typical Vaani Telugu copy on Drive (`vaani_telugu_transcribed/`).
- OpenSLR SLR66 (listing + a small decode sample).

**Read-only.** Nothing in the datasets is modified. Outputs go to
`Intelli-Speech-data/analysis_v2/audit_<date>/`. Transcript text is never printed or saved;
only counts, akshara totals and CER numbers derived from it.

**Before running:** open https://huggingface.co/datasets/ARTPARK-IISc/Vaani-Atypical-Speech-Corpus
while logged in and accept the access conditions. Run cells 1 → 5 in order.

In [ ]:
# ============================================================
# CELL 1 | HUGGING FACE LOGIN (your own account; the token stays in this runtime)
# ============================================================
# Interactive login with YOUR OWN account. The token is entered in the widget only;
# it is never printed, written to the repository, or passed to scripts.
from huggingface_hub import notebook_login, whoami
try:
    print("Already logged in as:", whoami()["name"])  # prints the username only
except Exception:
    notebook_login()

In [ ]:
# ============================================================
# CELL 2 | DRIVE MOUNT + REPOSITORY SYNC
# ============================================================
import os, subprocess, sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
REPO_DIR = Path("/content/Intelli-Speech")
if not (DRIVE_ROOT / "MyDrive").is_dir():
    from google.colab import drive
    drive.mount(str(DRIVE_ROOT))
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"], capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes in the Colab checkout (not pulling):\n" + dirty)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())
assert (REPO_DIR / "scripts" / "audit_v2.py").is_file(), "audit_v2.py missing: the v2 code is not pushed/pulled yet."

In [ ]:
# ============================================================
# CELL 3 | DEPENDENCIES (datasets only; nothing is downgraded)
# ============================================================
!pip install -q datasets soundfile
import datasets, numpy
print("datasets", datasets.__version__, "| numpy", numpy.__version__)

In [ ]:
# ============================================================
# CELL 4 | RUN THE AUDIT (read-only)
# The Telugu subset (~110 min) is cached on Drive so it downloads once.
# ============================================================
DATA_ROOT = "/content/drive/MyDrive/Intelli-Speech-data"
HF_CACHE = f"{DATA_ROOT}/hf_cache"
!python scripts/audit_v2.py --data-root "{DATA_ROOT}" --hf-cache "{HF_CACHE}"

In [ ]:
# ============================================================
# CELL 5 | RESULTS
# ============================================================
import json
from datetime import date
from pathlib import Path
import pandas as pd

OUT = sorted(Path(DATA_ROOT, "analysis_v2").glob("audit_*"))[-1]
summary = json.loads((OUT / "summary.json").read_text())
print("Output:", OUT)

for key in ["atypical_telugu", "typical_vaani"]:
    part = summary.get(key)
    if not part:
        continue
    print(f"\n=== {key} ===")
    for field in ["files", "decoded", "decode_errors", "speakers", "total_minutes", "duration_s",
                  "sample_rates", "channels", "clips_per_speaker", "files_with_near_full_scale_samples",
                  "effective_bandwidth_hz", "energy_dynamic_range_db"]:
        print(f"{field:38s} {part.get(field)}")
    print("duplicate audio groups:", len(part.get("duplicate_audio_groups", [])))

a = summary.get("atypical_telugu", {})
if a:
    print("\n=== atypical: labels and transcriptions ===")
    for field in ["difficulty_levels", "difficulty_speaker_cramers_v", "prompt_types", "distinct_prompts",
                  "prompts_shared_by_2plus_speakers", "intended_empty", "verbatim_empty",
                  "cer_verbatim_vs_intended", "max_abs_duration_mismatch_s"]:
        print(f"{field:38s} {a.get(field)}")
    print("\nDifficulty level by speaker (QC understandability, NOT clinical severity):")
    print(pd.DataFrame(a["difficulty_by_speaker"]).T.fillna(0).astype(int).to_string())
    print("\nPer-speaker table:")
    print(pd.read_csv(OUT / "atypical_per_speaker.csv").to_string(index=False))

if (OUT / "typical_per_speaker.csv").is_file():
    t = pd.read_csv(OUT / "typical_per_speaker.csv")
    print(f"\nTypical Vaani: {len(t)} speakers; clips per speaker:")
    print(t["clips"].describe().to_string())
    print("metadata columns:", summary["typical_vaani"].get("metadata_columns"))
    print("has prompt/image column:", summary["typical_vaani"].get("has_prompt_or_image_column"))

print("\n=== SLR66 ===")
print(json.dumps(summary.get("slr66"), indent=1, ensure_ascii=False)[:3000])

## After running

Save the notebook (Ctrl+S) so the outputs are stored, then tell Claude. The key questions
this audit answers:

1. Do we reproduce 189 clips / 7 speakers / ~109.1 min?
2. Is `difficulty_level` confounded with speaker (Cramér's V close to 1 → it mostly encodes speaker)?
3. Is `verbatim_transcription` populated enough for disfluency and syllable-rate analysis?
4. How many unique typical Vaani speakers do we have (decides whether to download more controls)?
5. Do channel descriptors (bandwidth, dynamic range) differ between atypical and typical data?